<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [2]</a>'.</span>

# Lamina Tubulin Biexponential Fit Workflow

Clean workflow for loading a calibrated HDF5 file, preparing aligned summed decays, generating per-pixel biexponential fit maps, and visualizing the effective lifetime result.


In [1]:
from pathlib import Path
import importlib
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = str(ROOT / "src")
if SRC in sys.path:
    sys.path.remove(SRC)
sys.path.insert(0, SRC)
MCS_FILE_SRC = ROOT.parent / "BrightEyes-MCS-File" / "src"
if MCS_FILE_SRC.exists():
    MCS_FILE_SRC = str(MCS_FILE_SRC)
    if MCS_FILE_SRC in sys.path:
        sys.path.remove(MCS_FILE_SRC)
    sys.path.insert(0, MCS_FILE_SRC)

import h5py
import matplotlib.pyplot as plt
import numpy as np

import brighteyes_mcs_file.alignment as alignment_module
import brighteyes_flim.graph_tools as graph
alignment_module = importlib.reload(alignment_module)
graph = importlib.reload(graph)

from brighteyes_mcs_file.alignment import Alignment
from brighteyes_mcs_file import show_h5_structure_html


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [2]:
fit_maps=np.load("fit_maps_biexponential.npz",allow_pickle=True)


FileNotFoundError: [Errno 2] No such file or directory: 'fit_maps_biexponential.npz'

## Visualize Effective Lifetime Fit


In [ ]:
tau1 = fit_maps["tau1"]
tau2 = fit_maps["tau1"] + fit_maps["delta_tau"]
a1 = np.clip(fit_maps["a1"], 0.0, 1.0)
tau = a1 * tau1 + (1.0 - a1) * tau2
intensity = fit_maps["intensity"]

tau1_ordered = tau1
tau2_ordered = tau2
a1_ordered = a1



thresholded_tau, thresholded_intensity, lifetime_mask = graph.threshold_lifetime_map(
    tau,
    intensity=intensity,
    threshold=0.05,
)

print("finite tau_eff pixels:", np.count_nonzero(np.isfinite(tau)))
print("finite tau1 pixels:", np.count_nonzero(np.isfinite(tau1)))
print("finite tau2 pixels:", np.count_nonzero(np.isfinite(tau2)))
print("thresholded pixels:", thresholded_tau.size)


In [ ]:
_=plt.hist2d(tau1.flatten(), tau2.flatten(), weights=(1-a1).flatten(),bins=100)

In [ ]:
_=plt.hist2d(tau2.flatten(), a1.flatten(), bins=100)

In [ ]:
v1=intensity*(a1)
vmin1=np.percentile(v1,0.05)
vmax1=np.percentile(v1,99.95)
v1_norm=(v1-vmin1)/(vmax1-vmin1)

v2=intensity*(1-a1)
vmin2=np.percentile(v2,0.05)
vmax2=np.percentile(v2,99.95)
v2_norm=(v2-vmin2)/(vmax2-vmin2)

a_norm=(a1-np.percentile(a1,0.05))/(np.percentile(a1,99.95)-np.percentile(a1,0.05))


plt.imshow(v2_norm + v1_norm, cmap="gray")
plt.imshow(a_norm, vmin=0, vmax=1, cmap="hsv", alpha=0.8)
plt.colorbar()


In [ ]:
_=plt.hist(a_norm.flatten(), bins=100)

In [ ]:
plt.imshow(fit_maps["intensity"], cmap="inferno")

In [ ]:
plt.imshow(a1, cmap="viridis")


In [ ]:
plt.imshow(intensity)

In [ ]:
# RGB component-intensity image: red = tau1 component, blue = tau2 component
component_tau1_intensity = a1 * intensity
component_tau2_intensity = (1.0 - a1) * intensity

component_mask = (
    lifetime_mask
    & np.isfinite(component_tau1_intensity)
    & np.isfinite(component_tau2_intensity)
)

valid_intensity = intensity[component_mask]
if valid_intensity.size:
    component_vmax = np.nanpercentile(valid_intensity, 99.5)
else:
    component_vmax = np.nanpercentile(intensity[np.isfinite(intensity)], 99.5)

if not np.isfinite(component_vmax) or component_vmax <= 0:
    component_vmax = np.nanmax(intensity)

rgb_components = np.zeros((*intensity.shape, 3), dtype=float)
rgb_components[..., 0] = np.clip(component_tau1_intensity / component_vmax, 0.0, 1.0)
rgb_components[..., 2] = np.clip(component_tau2_intensity / component_vmax, 0.0, 1.0)
rgb_components[~component_mask] = 0.0

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(rgb_components, origin="lower")
ax.set_title("Biexponential component intensity: tau1 red, tau2 blue")
ax.set_axis_off()
plt.show()


In [ ]:
graph.plot_lifetime_summary(
    intensity=intensity,
    lifetime=tau,
    pxsize=pixel_size_x_um,
    pxdwelltime=pxdwelltime,
    lifetime_bounds=[2.5, 4.5],
    crop=30,
    threshold=0.05,
    bins=500,
    colormap="turbo",
    weighted_histogram=True,
)


In [ ]:
graph.plot_equalized_lifetime_summary(
    intensity=intensity,
    lifetime=tau,
    pxsize=pixel_size_x_um,
    pxdwelltime=pxdwelltime,
    lifetime_bounds=[1.0, 8.0],
    crop=30,
    threshold=0.05,
    bins=500,
    colormap="turbo",
    equalization_reference=thresholded_tau,
    equalization_strength=4.0,
    equalization_bins=4096,
    colorbar_ticks=12,
)

